In [1]:
import psycopg2

conn = psycopg2.connect(
    host="localhost",
    database="postgres",
    user="postgres",
    password="",
    port=5432
)

cur = conn.cursor()

# 转化漏斗分析：View -> Cart -> Purchase
query_funnel = """
WITH user_product_events AS (
    SELECT 
        user_id,
        product_id,
        STRING_AGG(event_type, ',' ORDER BY event_time) as event_sequence
    FROM makeup_consumer_events.dec
    GROUP BY user_id, product_id
)
SELECT 
    -- View -> Cart 转化率
    COUNT(*) as total_with_view,
    SUM(CASE WHEN event_sequence ILIKE '%view%cart%' THEN 1 ELSE 0 END) as view_to_cart_count,
    ROUND(100.0 * SUM(CASE WHEN event_sequence ILIKE '%view%cart%' THEN 1 ELSE 0 END) 
          / COUNT(*), 2) as view_to_cart_rate_pct,
    
    -- Cart -> Purchase 转化率
    SUM(CASE WHEN event_sequence ILIKE '%cart%' THEN 1 ELSE 0 END) as total_with_cart,
    SUM(CASE WHEN event_sequence ILIKE '%cart%purchase%' THEN 1 ELSE 0 END) as cart_to_purchase_count,
    ROUND(100.0 * SUM(CASE WHEN event_sequence ILIKE '%cart%purchase%' THEN 1 ELSE 0 END) 
          / NULLIF(SUM(CASE WHEN event_sequence ILIKE '%cart%' THEN 1 ELSE 0 END), 0), 2) as cart_to_purchase_rate_pct
FROM user_product_events
WHERE event_sequence ILIKE '%view%';
"""

cur.execute(query_funnel)
result = cur.fetchone()

print("=" * 60)
print("转化漏斗分析结果")
print("=" * 60)
print(f"有浏览记录的用户-产品对: {result[0]}")
print(f"View → Cart 成功: {result[1]} 对")
print(f"View → Cart 转化率: {result[2]}%")
print()
print(f"有加购记录的用户-产品对: {result[3]}")
print(f"Cart → Purchase 成功: {result[4]} 对")
print(f"Cart → Purchase 转化率: {result[5]}%")
print("=" * 60)

cur.close()
conn.close()


转化漏斗分析结果
有浏览记录的用户-产品对: 1270675
View → Cart 成功: 223022 对
View → Cart 转化率: 17.55%

有加购记录的用户-产品对: 265959
Cart → Purchase 成功: 72041 对
Cart → Purchase 转化率: 27.09%
